In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "default"

trips = spark.table(f"{CATALOG}.{SCHEMA}.trips_clean")
zones = spark.table(f"{CATALOG}.{SCHEMA}.zones_clean")

print(f"Silver trip rows: {trips.count():,}")
print(f"Silver zone rows: {zones.count():,}")


In [0]:
dim_date = (
    trips.select(F.to_date("tpep_pickup_datetime").alias("full_date")).distinct()
    .union(trips.select(F.to_date("tpep_dropoff_datetime").alias("full_date")).distinct())
    .distinct()
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("day_of_week", F.date_format("full_date", "EEEE"))
    .withColumn("month", F.month("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("year", F.year("full_date"))
    .withColumn("is_weekend", F.dayofweek("full_date").isin([1, 7]))
    .select("date_key", "full_date", "day_of_week", "month", "quarter", "year", "is_weekend")
)

dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.dim_date")

print(f"dim_date rows: {dim_date.count()}")
display(dim_date.orderBy("full_date").limit(5))


In [0]:
##codechunk3
# dim_time — one row per hour of day
dim_time = spark.range(0, 24).withColumnRenamed("id", "hour") \
    .withColumn("time_key", F.col("hour")) \
    .withColumn("am_pm", F.when(F.col("hour") < 12, "AM").otherwise("PM")) \
    .withColumn(
        "time_of_day_bucket",
        F.when(F.col("hour").between(5, 9), "morning rush")
         .when(F.col("hour").between(10, 15), "midday")
         .when(F.col("hour").between(16, 19), "evening rush")
         .when(F.col("hour").between(20, 23), "evening")
         .otherwise("late night")
    ) \
    .select("time_key", "hour", "am_pm", "time_of_day_bucket")

dim_time.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.dim_time")

# dim_location — from cleaned zone lookup
dim_location = (
    zones
    .withColumnRenamed("location_id", "location_key")
    .select("location_key", "borough", "zone", "service_zone")
)
dim_location.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.dim_location")

# dim_rate_code, dim_payment_type, dim_vendor — TLC reference data
dim_rate_code = spark.createDataFrame([
    (1, "Standard rate"), (2, "JFK"), (3, "Newark"),
    (4, "Nassau or Westchester"), (5, "Negotiated fare"), (6, "Group ride"),
], ["rate_code_key", "description"])
dim_rate_code.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.dim_rate_code")

dim_payment_type = spark.createDataFrame([
    (1, "Credit card"), (2, "Cash"), (3, "No charge"),
    (4, "Dispute"), (5, "Unknown"), (6, "Voided trip"),
], ["payment_type_key", "description"])
dim_payment_type.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.dim_payment_type")

dim_vendor = spark.createDataFrame([
    (1, "Creative Mobile Technologies"), (2, "VeriFone Inc."),
], ["vendor_key", "vendor_name"])
dim_vendor.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.dim_vendor")

print("All dimension tables built.")

In [0]:
fact_trips = (
    trips
    .withColumn("pickup_date_key", F.date_format("tpep_pickup_datetime", "yyyyMMdd").cast("int"))
    .withColumn("pickup_time_key", F.hour("tpep_pickup_datetime"))
    .select(
        F.monotonically_increasing_id().alias("trip_id"),
        "pickup_date_key",
        "pickup_time_key",
        F.col("PULocationID").alias("pickup_location_key"),
        F.col("DOLocationID").alias("dropoff_location_key"),
        F.col("RatecodeID").cast("int").alias("rate_code_key"),
        F.col("payment_type").cast("int").alias("payment_type_key"),
        F.col("VendorID").cast("int").alias("vendor_key"),
        "passenger_count",
        "trip_distance",
        "fare_amount",
        "tip_amount",
        "tolls_amount",
        "total_amount",
        F.round(F.col("trip_duration_minutes")).cast("int").alias("trip_duration_minutes"),
    )
)

fact_trips.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.fact_trips")

print(f"fact_trips rows: {fact_trips.count():,}")
